<a href="https://colab.research.google.com/github/ben-blake/umkc-deep-learning/blob/main/hw3-transformer/HW3-formal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework 3 - Attention and Transformer
- Classify the speakers of given features.
- Main goal: Learn how to use transformer.

**Name:** Ben Blake

**Student ID:** 14387365

# Data

## Dataset
- Original dataset is [Voxceleb1](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/).
- The [license](https://creativecommons.org/licenses/by/4.0/) and [complete version](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/files/license.txt) of Voxceleb1.
- We randomly select 600 speakers from Voxceleb1.
- Then preprocess the raw waveforms into mel-spectrograms.

- Args:
  - data_dir: The path to the data directory.
  - metadata_path: The path to the metadata.
  - segment_len: The length of audio segment for training.
- The architecture of data directory \\
  - data directory \\
  |---- metadata.json \\
  |---- testdata.json \\
  |---- mapping.json \\
  |---- uttr-{random string}.pt \\

- The information in metadata
  - "n_mels": The dimention of mel-spectrogram.
  - "speakers": A dictionary.
    - Key: speaker ids.
    - value: "feature_path" and "mel_len"


For efficiency, we segment the mel-spectrograms into segments in the traing step.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/UMKC/Fall 2026/Deep Learning/HW3/Dataset_HW3.zip" "./Dataset_HW3.zip"
!unzip -q Dataset_HW3.zip
!ls

Mounted at /content/drive
Dataset  Dataset_HW3.zip  drive  sample_data


In [ ]:
import os
import json
import torch
import random
from pathlib import Path
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


class myDataset(Dataset):
  def __init__(self, data_dir, segment_len=128):
    self.data_dir = data_dir
    self.segment_len = segment_len

    # Load the mapping from speaker neme to their corresponding id.
    mapping_path = Path(data_dir) / "mapping.json"
    mapping = json.load(mapping_path.open())
    self.speaker2id = mapping["speaker2id"]

    # Load metadata of training data.
    metadata_path = Path(data_dir) / "metadata.json"
    metadata = json.load(open(metadata_path))["speakers"]

    # Get the total number of speaker.
    self.speaker_num = len(metadata.keys())
    self.data = []
    for speaker in metadata.keys():
      for utterances in metadata[speaker]:
        self.data.append([utterances["feature_path"], self.speaker2id[speaker]])

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    feat_path, speaker = self.data[index]
    # Load preprocessed mel-spectrogram.
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    # Segmemt mel-spectrogram into "segment_len" frames.
    if len(mel) > self.segment_len:
      # Randomly get the starting point of the segment.
      start = random.randint(0, len(mel) - self.segment_len)
      # Get a segment with "segment_len" frames.
      mel = torch.FloatTensor(mel[start:start+self.segment_len])
    else:
      mel = torch.FloatTensor(mel)
    # Turn the speaker id into long for computing loss later.
    speaker = torch.FloatTensor([speaker]).long()
    return mel, speaker

  def get_speaker_number(self):
    return self.speaker_num

## Dataloader (2pts)
- Split dataset into training dataset(90%) and validation dataset(10%).
- Create dataloader to iterate the data.


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence


def collate_batch(batch):
  # Process features within a batch.
  """Collate a batch of data."""
  mel, speaker = zip(*batch)
  # Because we train the model batch by batch, we need to pad the features in the same batch to make their lengths the same.
  mel = pad_sequence(mel, batch_first=True, padding_value=-20)    # pad log 10^(-20) which is very small value.
  # mel: (batch size, length, 40)
  return mel, torch.FloatTensor(speaker).long()


def get_dataloader(data_dir, batch_size, n_workers):
  """Generate dataloader"""
  dataset = myDataset(data_dir)
  speaker_num = dataset.get_speaker_number()
  # Split dataset into training dataset and validation dataset
  trainlen = int(0.9 * len(dataset))
  lengths = [trainlen, len(dataset) - trainlen]
  trainset, validset = random_split(dataset, lengths)

# (1pt)TODO: Define Training Data Loader and set batch_size=batch_size,shuffle=True,drop_last=True,num_workers=n_workers,pin_memory=True and collate_fn=collate_batch
  # fixed the class name (DataLoader, capital L) and swapped the "and" between the last two kwargs
  # for a comma, it was a syntax error. collate_batch pads anything shorter than the 128-frame crop
  # so the batch stacks into one tensor
  train_loader = DataLoader(
      trainset,
      batch_size=batch_size,shuffle=True,drop_last=True,num_workers=n_workers,pin_memory=True,collate_fn=collate_batch
  )
  # (1pt)TODO: Define Validation Data Loader and set batch_size=batch_size,drop_last=True,num_workers=n_workers,pin_memory=True and collate_fn=collate_batch
  # same fixes as above. no shuffle here since order doesn't matter for scoring the valid set
  valid_loader = DataLoader(
      validset,
      batch_size=batch_size,drop_last=True,num_workers=n_workers,pin_memory=True,collate_fn=collate_batch
  )

  return train_loader, valid_loader, speaker_num


# Model (5pts)
- TransformerEncoderLayer:
  - Base transformer encoder layer in [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
  - Parameters:
    - d_model: the number of expected features of the input (required).

    - nhead: the number of heads of the multiheadattention models (required).

    - dim_feedforward: the dimension of the feedforward network model (default=2048).

    - dropout: the dropout value (default=0.1).

    - activation: the activation function of intermediate layer, relu or gelu (default=relu).

- TransformerEncoder:
  - TransformerEncoder is a stack of N transformer encoder layers
  - Parameters:
    - encoder_layer: an instance of the TransformerEncoderLayer() class (required).

    - num_layers: the number of sub-encoder-layers in the encoder (required).

    - norm: the layer normalization component (optional).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Classifier(nn.Module):
  def __init__(self, d_model=80, n_spks=600, dropout=0.1):
    super().__init__()
    # Project the dimension of features from that of input into d_model.
    self.prenet = nn.Linear(40, d_model) # (1pt)TODO: Linear layer from 40 to d_model
    # maps each 40-dim mel frame up to the transformer's d_model=80

    # (1pt)TODO: Use TransformerEncoderLayer as encoder_layer and set dim_feedforward=256 and nhead=2
    # was nn.TransformerEncoder, which is the stack wrapper and wants an encoder_layer + num_layers.
    # we want a single layer here. batch_first is left at its default (False), which is why forward
    # permutes to (length, batch, d_model) first. 80 / 2 heads = 40 dims per head
    self.encoder_layer = nn.TransformerEncoderLayer(
        d_model=d_model, dim_feedforward=256, nhead=2
    )
    # self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=2)

    # Project the the dimension of features from d_model into speaker nums.
    self.pred_layer = nn.Sequential(
      nn.Linear(d_model, d_model), # (1pt)TODO: Linear layer from d_model to d_model
      # added the missing comma, without it the Sequential doesn't parse
      nn.ReLU(), # (1pt)TODO: Relu layer
      # the class is nn.ReLU, nn.Relu doesn't exist
      nn.Linear(d_model, n_spks), # (1pt)TODO: Linear layer from d_model to n_spks
      # one logit per speaker; no softmax since CrossEntropyLoss handles that
    )

  def forward(self, mels):
    """
    args:
      mels: (batch size, length, 40)
    return:
      out: (batch size, n_spks)
    """
    # out: (batch size, length, d_model)
    out = self.prenet(mels)
    # out: (length, batch size, d_model)
    out = out.permute(1, 0, 2)
    # The encoder layer expect features in the shape of (length, batch size, d_model).
    out = self.encoder_layer(out)
    # out: (batch size, length, d_model)
    out = out.transpose(0, 1)
    # mean pooling
    stats = out.mean(dim=1)

    # out: (batch, n_spks)
    out = self.pred_layer(stats)
    return out


# Learning rate schedule
- For transformer architecture, the design of learning rate schedule is different from that of CNN.
- Previous works show that the warmup of learning rate is useful for training models with transformer architectures.
- The warmup schedule
  - Set learning rate to 0 in the beginning.
  - The learning rate increases linearly from 0 to initial learning rate during warmup period.

In [ ]:
import math

import torch
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR


def get_cosine_schedule_with_warmup(
  optimizer: Optimizer,
  num_warmup_steps: int,
  num_training_steps: int,
  num_cycles: float = 0.5,
  last_epoch: int = -1,
):
  """
  Create a schedule with a learning rate that decreases following the values of the cosine function between the
  initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
  initial lr set in the optimizer.

  Args:
    optimizer (:class:`~torch.optim.Optimizer`):
      The optimizer for which to schedule the learning rate.
    num_warmup_steps (:obj:`int`):
      The number of steps for the warmup phase.
    num_training_steps (:obj:`int`):
      The total number of training steps.
    num_cycles (:obj:`float`, `optional`, defaults to 0.5):
      The number of waves in the cosine schedule (the defaults is to just decrease from the max value to 0
      following a half-cosine).
    last_epoch (:obj:`int`, `optional`, defaults to -1):
      The index of the last epoch when resuming training.

  Return:
    :obj:`torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
  """

  def lr_lambda(current_step):
    # Warmup
    if current_step < num_warmup_steps:
      return float(current_step) / float(max(1, num_warmup_steps))
    # decadence
    progress = float(current_step - num_warmup_steps) / float(
      max(1, num_training_steps - num_warmup_steps)
    )
    return max(
      0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
    )

  return LambdaLR(optimizer, lr_lambda, last_epoch)


# Model Function (1pt)
- Model forward function.

In [ ]:
import torch


def model_fn(batch, model, criterion, device):
  """Forward a batch through the model."""

  mels, labels = batch
  mels = mels.to(device)
  labels = labels.to(device)

  outs = model(mels)

  loss = criterion(outs, labels) #(1pt)TODO: Use criterion to calculate the loss
  # outs are raw logits (batch, n_spks), labels are class indices (batch,), which is what CrossEntropyLoss wants

  # Get the speaker id with highest probability.
  preds = outs.argmax(1)
  # Compute accuracy.
  accuracy = torch.mean((preds == labels).float())

  return loss, accuracy


# Validate (3pts)
- Calculate accuracy of the validation set.

In [ ]:
from tqdm import tqdm
import torch


def valid(dataloader, model, criterion, device):
  """Validate on validation set."""

  model.eval()
  running_loss = 0.0
  running_accuracy = 0.0
  pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit=" uttr")

  for i, batch in enumerate(dataloader):
    with torch.no_grad():
      loss, accuracy = model_fn(batch, model, criterion, device) #(1pt)TODO: Use model_fn function above to calculate the loss and accuracy
      # same forward pass as training, just under no_grad so no graph gets built
      running_loss += loss.item() #(1pt)TODO: Get cumulative loss
      # .item() pulls out a python float so we're not holding onto GPU tensors
      running_accuracy += accuracy.item() #(1pt)TODO: Get cumulative Accuracy
      # these are per-batch means. drop_last keeps every batch at 32 (6944 / 32 = 217, so nothing
      # actually gets dropped), which makes dividing by the batch count below the true accuracy

    pbar.update(dataloader.batch_size)
    pbar.set_postfix(
      loss=f"{running_loss / (i+1):.2f}",
      accuracy=f"{running_accuracy / (i+1):.2f}",
    )

  pbar.close()
  model.train()

  return running_accuracy / len(dataloader)


# Main function (5pts)

In [ ]:
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split

# You may change the total_steps to a larger number ie. 70000 for a higher accuracy
# It may take up to 3 minutes for every 2000 steps.
def parse_args():
  """arguments"""
  config = {
    "data_dir": "./Dataset",
    "save_path": "model.ckpt",
    "batch_size": 32,
    "n_workers": 8,
    "valid_steps": 2000,
    "warmup_steps": 1000,
    "save_steps": 10000,
    "total_steps": 20000,
  }

  return config


def main(
  data_dir,
  save_path,
  batch_size,
  n_workers,
  valid_steps,
  warmup_steps,
  total_steps,
  save_steps,
):
  """Main function."""
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"[Info]: Use {device} now!")

  train_loader, valid_loader, speaker_num = get_dataloader(data_dir, batch_size, n_workers) #(1pt)TODO: Use get_dataloader function above to get data loaders.
  # does the 90/10 split and hands back speaker_num too, so the classifier head sizes itself off the data
  train_iterator = iter(train_loader)
  print(f"[Info]: Finish loading data!",flush = True)

  model = Classifier(n_spks=speaker_num).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = AdamW(model.parameters(), lr=1e-3) #(1pt)TODO: Use a proper optimizer and set a proper learning rate
  # AdamW is already imported above and is the usual pick for transformers. 1e-3 is the peak lr;
  # the scheduler ramps up to it and decays down from it
  scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps) #(1pt)TODO: Use scheduler defined above
  # linear warmup over the first 1000 steps, then half-cosine decay to 0 by total_steps.
  # it's stepped once per batch in the loop, so both counts are in steps, not epochs
  print(f"[Info]: Finish creating model!",flush = True)

  best_accuracy = -1.0
  best_state_dict = None

  pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

  for step in range(total_steps):
    # Get data
    try:
      batch = next(train_iterator)
    except StopIteration:
      train_iterator = iter(train_loader)
      batch = next(train_iterator)

    loss, accuracy = model_fn(batch, model, criterion, device) #(1pt)TODO: Use model_fn function above to calculate the loss and accuracy
    # forward pass on the training batch; loss keeps its graph so backward() below works
    batch_loss = loss.item()
    batch_accuracy = accuracy.item()

    # Updata model
    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    # Log
    pbar.update()
    pbar.set_postfix(
      loss=f"{batch_loss:.2f}",
      accuracy=f"{batch_accuracy:.2f}",
      step=step + 1,
    )

    # Do validation
    if (step + 1) % valid_steps == 0:
      pbar.close()

      valid_accuracy = valid(valid_loader, model, criterion, device) #(1pt)TODO: Use valid function above to calculate the vaild_accuracy
      # average accuracy over the held-out 10%; valid() flips eval()/train() on its own

      # keep the best model
      if valid_accuracy > best_accuracy:
        best_accuracy = valid_accuracy
        best_state_dict = model.state_dict()

      pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

    # Save the best model so far.
    if (step + 1) % save_steps == 0 and best_state_dict is not None:
      torch.save(best_state_dict, save_path)
      pbar.write(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")

  pbar.close()


if __name__ == "__main__":
  main(**parse_args())


[Info]: Use cuda now!


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Info]: Finish loading data!
[Info]: Finish creating model!


Train:  98% 1952/2000 [01:36<00:01, 29.20 step/s, accuracy=0.22, loss=4.00, step=1952]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Train: 100% 2000/2000 [01:38<00:00, 20.40 step/s, accuracy=0.22, loss=3.62, step=2000]
Valid: 100% 6944/6944 [00:10<00:00, 643.75 uttr/s, accuracy=0.20, loss=3.88]
Train: 100% 2000/2000 [01:05<00:00, 30.33 step/s, accuracy=0.34, loss=3.54, step=4000]
Valid: 100% 6944/6944 [00:08<00:00, 866.62 uttr/s, accuracy=0.31, loss=3.16] 
Train: 100% 2000/2000 [01:16<00:00, 26.02 step/s, accuracy=0.38, loss=2.90, step=6000]
Valid: 100% 6944/69

Step 10000, best model saved. (accuracy=0.4456)


Train: 100% 2000/2000 [01:07<00:00, 29.76 step/s, accuracy=0.59, loss=1.87, step=12000]
Valid: 100% 6944/6944 [00:08<00:00, 826.79 uttr/s, accuracy=0.48, loss=2.32]
Train: 100% 2000/2000 [01:13<00:00, 27.30 step/s, accuracy=0.50, loss=2.23, step=14000]
Valid: 100% 6944/6944 [00:09<00:00, 699.12 uttr/s, accuracy=0.50, loss=2.23]
Train: 100% 2000/2000 [01:08<00:00, 29.38 step/s, accuracy=0.53, loss=1.90, step=16000]
Valid: 100% 6944/6944 [00:09<00:00, 732.71 uttr/s, accuracy=0.52, loss=2.15]
Train: 100% 2000/2000 [01:09<00:00, 28.80 step/s, accuracy=0.47, loss=2.05, step=18000]
Valid: 100% 6944/6944 [00:09<00:00, 701.01 uttr/s, accuracy=0.53, loss=2.08]
Train: 100% 2000/2000 [01:08<00:00, 29.39 step/s, accuracy=0.34, loss=2.18, step=2e+4]
Valid: 100% 6944/6944 [00:09<00:00, 753.74 uttr/s, accuracy=0.52, loss=2.12] 
Train:   0% 0/2000 [00:00<?, ? step/s]


Step 20000, best model saved. (accuracy=0.5291)


# **Summary**

I trained the transformer speaker classifier for the default 20,000 steps on a T4. Best validation accuracy was **0.5291**, at step 18,000. There are 600 speakers, so random guessing would get about 0.17%.

**Data**

The dataset has 69,438 utterances from 600 speakers, and the 90/10 split gives 62,494 for training and 6,944 for validation. Each utterance is a sequence of 40-dim mel frames, and `myDataset` grabs a random 128-frame chunk out of each one. Only one utterance in the whole set is shorter than 128 frames (shortest is 96, median is 584), so the -20 padding in `collate_batch` basically never comes into play. With a batch size of 32, one epoch is 1,952 steps, so 20k steps is a little over 10 epochs.

**Model**

Each 40-dim frame goes through the prenet (Linear 40 to 80), then a single TransformerEncoderLayer with d_model 80, 2 heads (40 dims each), and a 256-wide feedforward. The output gets mean-pooled over time and goes through Linear(80, 80), ReLU, Linear(80, 600) to get one logit per speaker. It's a pretty small model, 125,896 parameters total: 3,280 in the prenet, 67,536 in the encoder layer, and 55,080 in the head. The encoder layer uses the default `batch_first=False`, which is why `forward` permutes to (length, batch, d_model) before it and transposes back after.

**Training**

CrossEntropyLoss with AdamW at a peak lr of 1e-3. The scheduler ramps the lr up linearly over the first 1,000 steps, then decays it to 0 along a half-cosine by step 20,000. Validation ran every 2,000 steps:

| step | valid acc | valid loss |
|---|---|---|
| 2,000 | 0.20 | 3.88 |
| 4,000 | 0.31 | 3.16 |
| 6,000 | 0.36 | 2.89 |
| 8,000 | 0.41 | 2.69 |
| 10,000 | 0.45 | 2.47 |
| 12,000 | 0.48 | 2.32 |
| 14,000 | 0.50 | 2.23 |
| 16,000 | 0.52 | 2.15 |
| 18,000 | **0.53** | 2.08 |
| 20,000 | 0.52 | 2.12 |

**What the results show**

Validation accuracy went up and validation loss went down at every checkpoint through step 18,000. The small drop at 20,000 (0.53 to 0.52) is most likely just noise. By step 18k the cosine schedule has the lr down to about 2.7e-5, so the weights barely change over the last 2k steps. The validation set also uses random 128-frame crops, so its numbers wobble a bit from one pass to the next.

The model is underfitting, not overfitting. Training batch accuracy (one noisy batch of 32) stays in about the same range as validation the whole run, and there's no point where validation gets worse while training keeps improving. So what's holding it back is model size and training length. The TODOs lock in the architecture, and the template says more steps (like 70,000) would give higher accuracy, but `total_steps` isn't a TODO line, so I left it at 20,000.

**A couple of things I noticed in the provided code**

- `best_state_dict = model.state_dict()` doesn't make a copy. The state dict tensors share memory with the model's live weights (I checked this). So the `model.ckpt` saved at step 20,000 actually holds the step-20,000 weights (0.52 valid), not the step-18,000 weights the log reports as 0.5291. It's only about a 0.01 gap here, and fixing it would mean adding a `copy.deepcopy`, which is outside the TODOs.
- `n_workers: 8` throws a PyTorch warning because the Colab runtime only has 2 CPUs. It didn't cause any problems; training still ran at about 30 steps/s.

#**AI Use Disclosure**

Anthropic Claude Code was utilized on this assignment. Its contributions were: assistance with the explanatory comments next to updated TODO items and the summary. All code outside the TODO markers is the provided template, unmodified apart from the Google Drive mount / dataset unzip cell. I personally modified, reviewed, and ran the final notebook.

#**Documentation Referenced**

**Data loading:**
- torch.utils.data.DataLoader - https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
- torch.utils.data.random_split - https://pytorch.org/docs/stable/data.html#torch.utils.data.random_split
- pad_sequence - https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html

**Model:**
- nn.TransformerEncoderLayer - https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html
- nn.TransformerEncoder - https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html
- nn.Linear - https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
- nn.ReLU - https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html
- Attention Is All You Need - https://arxiv.org/abs/1706.03762

**Loss, training & evaluation:**
- nn.CrossEntropyLoss - https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
- torch.optim.AdamW - https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html
- LambdaLR (warmup + cosine schedule) - https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LambdaLR.html
- torch.no_grad - https://pytorch.org/docs/stable/generated/torch.no_grad.html
- Tensor.item - https://pytorch.org/docs/stable/generated/torch.Tensor.item.html